# Part 1 — Global Localization (Perfect Situation)

## Objective
Show that the relative pose between two robots can be easily computed when both are tracked in a shared reference frame.

### Theory
Given global poses $^O T_A$ and $^O T_B$ from a vicon/GT tracker:
$$
^A T_B = (^O T_A)^{-1} \cdot ^O T_B
$$

---

In [ ]:
# Setup: imports & reproducible seed
import sys; sys.path.insert(0, '/workspace/Python/Relative-localisation-tutorial')

import numpy as np
from matplotlib import pyplot as plt
from arp_utility_code.UtilityCode.Manifpy_helper import sgal3_to_se3_matrix 

from src.config import GlobalLocalizationConfig
from src.trajectories import build_two_robot_scenario
from src.relative_pose import compute_relative_pose, compute_noisy_relative_pose, display_relative_pose
from src.plots import plot_global_trajectory_with_relative_arrow, plot_perfect_vs_noisy_comparison

np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)
print('Imports complete')

### Parameters
---
Configure simulation parameters below:

In [ ]:
config = GlobalLocalizationConfig(time=10.0, dt=0.1)
print(f'Simulation: {int(config.time)}s @ {config.dt}s timestep ({len(np.arange(0, config.time, config.dt))} steps)')

### Build Scenario
---

In [ ]:
scenario = build_two_robot_scenario(config)
traj_a = scenario['gt_traj_a']
traj_b = scenario['gt_traj_b']
rel_offset = scenario['rel_offset']

print(f'Robot A start: {sgal3_to_se3_matrix(traj_a.X_OR)[:3, 3]}')
print(f'Robot B start: {sgal3_to_se3_matrix(traj_b.X_OR)[:3, 3]}')
print(f'Relative offset (desired): {rel_offset}')

### Perfect Relative Pose
---

In [ ]:
# Compute and display perfect relative pose
result_perfect = compute_relative_pose(traj_a, traj_b)
display_relative_pose(result_perfect, rel_offset, num_rows=8)

### Verify
---

In [ ]:
pos_err = np.linalg.norm(result_perfect.rel_pos - rel_offset, axis=1)
rot_err = np.linalg.norm(result_perfect.rel_orient, axis=1)

print('=== Verification ===')
print(f'Position error max:  {pos_err.max():.2e} m')
print(f'Orientation error:   {rot_err.max():.2e} rad')
print(f'Relative pos norm constant? {np.allclose(np.linalg.norm(result_perfect.rel_pos, axis=1), np.linalg.norm(rel_offset))}')

### Visualize: Trajectories + Relative Pose Arrows
---

In [ ]:
# One-call figure with GT paths and green relative-pose arrows in XY panel
plot_global_trajectory_with_relative_arrow(
    traj_a, traj_b, rel_result=result_perfect,
    labels=("Robot A", "Robot B"), n_arrows=8
).show()

### Noisy Relative Pose
---

In [ ]:
# Compute noisy case & error vs perfect
np.random.seed(42)
result_noisy = compute_noisy_relative_pose(traj_a, traj_b, sig_pos=0.01, sig_orient=0.05)
disp_err = np.linalg.norm(result_noisy.rel_pos - result_perfect.rel_pos, axis=1)

# Positions for left-panel arrow start points
N = len(result_perfect.time)
_pos = lambda t: np.array([sgal3_to_se3_matrix(t.X_OR.compose(t.X_RRi[i]))[:3, 3] for i in range(N)])
pos_a_arr, pos_b_arr = _pos(traj_a), _pos(traj_b)

# Plot noisy vs perfect comparison
plot_perfect_vs_noisy_comparison(
    result_perfect, result_noisy,
    pos_a=pos_a_arr,
    pos_b=pos_b_arr[:N],
    n_arrows=8,
).show()

---
### Summary
- In the perfect case, relative pose computation is exact: $^A T_B$ has constant norm.
- With vicon noise (σ=1 cm pos, σ=0.02 rad orient), errors are small but noticeable.
- Next Part: Analyze how sensor geometry affects GDOP for UWB-based relative localisation when perfect GT is unavailable.